In [ ]:
%reload_ext autoreload
%autoreload 2

import pandas as pd
import altair as alt
from preprocessing import create_clean_dataframe
from basic_eda import *
import numpy as np

# Import Data

In [ ]:
path = "../data/simpsons_episodes.csv"

# Data Preprocessing

In [ ]:
df = create_clean_dataframe(path)

# Data Quality checks

## Null values
The dataset contains a small number of missing values. Specifically, `imdb_rating` has 3 missing entries, `us_viewers_in_millions` has 6 missing values, and `views` has 4 missing values. All other columns are complete with no null values.


In [ ]:
print(df.info())

## Unique values
An inspection of the unique values reveals some interesting patterns in the dataset. There are 28 unique seasons and the maximum episode number within a season is 25, which is consistent with typical TV season structures. The `original_air_date` column contains 591 unique dates out of 600 episodes, meaning that 9 episodes share the same air date. The `day_aired` variable has only 5 unique values and is highly unbalanced: most episodes were aired on Sunday (506), followed by Thursday (89), while Tuesday and Wednesday only appear twice each and Friday only once. This strong imbalance suggests that the show was predominantly broadcast on a single day of the week.


In [ ]:
print_unique_values(df)
df["day_aired"].value_counts()

## Unique values 2

no té sentit mirar als uniques de imdb_rating, views, is_last_episode us_viewers_in_millions

Contem el numero de capitols per temporada

In [ ]:
df.groupby("season").size().reset_index(name="episode_count").sort_values("season")

## Outlier detection
An outlier detection analysis was performed using the IQR method on the numerical variables `imdb_rating`, `us_viewers_in_millions`, and `views`. The results show a very small number of outliers in `imdb_rating` (2) and a slightly higher number in `us_viewers_in_millions` (9). In contrast, the `views` variable contains a larger number of potential outliers (41), suggesting a higher variability in this metric. These values should be considered during the analysis, as extreme observations could influence some visualizations or statistical interpretations.

In [ ]:
columns_to_check = ["imdb_rating", "us_viewers_in_millions", "views"]
print_outliers(df, columns_to_check)

## Outlier Detection 2

jo em petava la variable de views així ens estalviem una. A part no és interessant

### IMDB Rating

In [ ]:
outliers_imdb_rating = count_outliers_iqr(df, "imdb_rating")
lb = outliers_imdb_rating[2]
ub = outliers_imdb_rating[3]
outliers_imdb_rating[0]

We find 2 outliers belonging to very low ratings. This are:
* The 11th episode of season 9
* The 22th episode of season 11

Since we don't have many of them and the range of values 0-10 of the variable is not very large, keeping them won't affect visualizations and we decide to do so

In [ ]:
base = alt.Chart(df).transform_calculate(
    outlier=f"(datum.imdb_rating < {lb}) || (datum.imdb_rating > {ub})"
)

hist = base.mark_bar().encode(
    x=alt.X("imdb_rating:Q", bin=alt.Bin(maxbins=30), title="IMDb rating"),
    y=alt.Y("count():Q", title="Episodes"),
    color=alt.condition("datum.outlier", alt.value("red"), alt.value("steelblue"))
)

bounds_df = pd.DataFrame({"bound": [lb, ub]})
bounds = alt.Chart(bounds_df).mark_rule(color="black", strokeDash=[4, 4]).encode(
    x="bound:Q"
 )

chart = (hist + bounds).properties(
    title="Distribution of IMDb Ratings with Outliers Highlighted"
 )
chart

# Us Viewers

We find 9 outliers on the variable us_viewers_in_millions, all of them belonging to early seasons of the show. Some sepcial events happened on these dates which may have affected the show's audience:
* On March 25, 1990, a tragic arson attack at the Happy Land social club in the Bronx, New York City, killed 87 people
* On April 29, 1990, the US space shuttle Discovery (STS-31) returned to Earth after deploying the Hubble Space Telescope. Major events included the LA Lakers defeating the Houston Rockets to set a record, Nirvana playing a legendary show in Washington, D.C., and Dan Quisenberry announcing his MLB retirement. 
* https://www.onthisday.com/date/1990/october/11
* The world's very first text message wasn't a love note, a meme, or a status update—it was a simple holiday greeting. On December 3, 1992, British engineer Neil Papworth sent the message “Merry Christmas” from his computer to the mobile phone of Vodafone director Richard Jarvis.

In [ ]:
outliers_viewers = count_outliers_iqr(df, "us_viewers_in_millions")
lb = outliers_viewers[2]
ub = outliers_viewers[3]
outliers_viewers_df = outliers_viewers[0]
outliers_viewers_df['season'].value_counts()

In [ ]:
# Ensure IQR bounds for viewers exist
if "lb" not in globals() or "ub" not in globals():
    outliers_viewers = count_outliers_iqr(df, "us_viewers_in_millions")
    lb = outliers_viewers[2]
    ub = outliers_viewers[3]

first_seasons = df[df["season"] < 5].copy()
first_seasons = first_seasons.sort_values(["season", "original_air_date"])

# Connect lines by removing null viewer values
line = (
    alt.Chart(first_seasons)
    .transform_filter("isValid(datum.us_viewers_in_millions)")
    .mark_line()
    .encode(
        x=alt.X("original_air_date:T", title="Original Air Date"),
        y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (Millions)"),
        color=alt.Color("season:N", title="Season")
    )
)

# Vertical separators at the last episode of each season
season_separators = alt.Chart(
    first_seasons[first_seasons["is_last_episode"] == 1]
).mark_rule(color="black", strokeDash=[4, 4], opacity=0.6).encode(
    x=alt.X("original_air_date:T")
)

# Horizontal lower/upper bounds
bounds_df = pd.DataFrame(
    {
        "bound": [lb, ub],
        "label": ["Lower bound", "Upper bound"]
    }
)
bounds = alt.Chart(bounds_df).mark_rule(color="firebrick", strokeDash=[6, 4]).encode(
    y=alt.Y("bound:Q"),
    detail="label:N",
    tooltip=["label:N", alt.Tooltip("bound:Q", format=".2f")]
 )

chart = (line + season_separators + bounds).properties(
    title="US Viewers Distribution for First Seasons with Season Separators and IQR Bounds"
 )

chart

In [ ]:
# density plot for viewers in all seasons with lines indifcating lower and upper bounds for outliers
chart = alt.Chart(df).transform_density(
    "us_viewers_in_millions",
    as_=["us_viewers_in_millions", "density"]
).mark_area(opacity=0.5, color="steelblue").encode(
    x=alt.X("us_viewers_in_millions:Q", title="US Viewers (Millions)"),
    y=alt.Y("density:Q", title="Density")
)

bounds = alt.Chart(bounds_df).mark_rule(color="firebrick", strokeDash=[6, 4]).encode(
    x=alt.X("bound:Q"),
    detail="label:N",
    tooltip=["label:N", alt.Tooltip("bound:Q", format=".2f")]
)

(chart + bounds)

In [ ]:
alt.Chart(df).mark_bar().encode(
    x='day_aired',
    y='average(imdb_rating)'
)

The outliers are coaused of the decrease in the show's popularity. We'll keep them in the distribution

In [ ]:
alt.Chart(df).mark_bar().encode(
    x='day_aired',
    y='count()'
)

In [ ]:
df.info()

## imdb_rating

The missing ratings from this variable come from the last season of the dataset, where there are only 4 episodes and 3 present in the data.

In [ ]:
alt.Chart(df).mark_bar().encode(
    x='season',
    y='count()'
)

# us_viewers_in_millions

3 of the 6 missing values of this variable come from the espisodes of season 28, The other 3 come from episodes of season 8

In [ ]:
df[df['us_viewers_in_millions'].isna()]

# Decision

We detected that there are null values, so we found another data source that had the ratings of the three missing episodes.aving information of 4 more points won't help us to identifying trends any better compared to all the seasons we have more episodes of. We decide to eliminate epsiodes from season 28 and still have data of the seasons 1-27, which will be enough to answer the questions

In order to not have 3 empyt points for season 8 and make the visualkization akward, we can use other sources to fins the audience values for the missing episoed and plug them into our dataset.

In [ ]:
[df['season'] == 8]

we will use another data source to plug in the missing values for the episodes in season 8
https://www.kaggle.com/datasets/jonbown/simpsons-episodes-2016?select=simpsons_episodes.csv

In [ ]:
dataset2 = pd.read_csv('../data/simpsons_episodes_second_source.csv')
dataset2[dataset2['season'] == 8]

In [ ]:
df2 = df.merge(dataset2[['season', 'number_in_season', 'us_viewers_in_millions']], on=['season', 'number_in_season'], how='left', suffixes=('', '_from_ds2'))
df2['us_viewers_in_millions'] = df2['us_viewers_in_millions'].fillna(df2['us_viewers_in_millions_from_ds2'])
df2.drop(columns=['us_viewers_in_millions_from_ds2'], inplace=True)

In [ ]:
# Q1
alt.Chart(df).mark_line().encode(
    x='original_air_date',
    y='average(imdb_rating)'
)

In [ ]:
# Q2
alt.Chart(df).mark_line().encode(
    x='original_air_date',
    y='average(us_viewers_in_millions)'
)

In [ ]:
# Q3
alt.Chart(df).mark_circle().encode(
    x='us_viewers_in_millions',
    y='imdb_rating'
)

In [ ]:
# Q4
# Day Rating Distribution
alt.Chart(df).mark_bar().encode(
    x='day_aired',
    y='average(imdb_rating)'
)